In [2]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'

out_dir = "/mnt/data/smooth_constraint_svgs"
os.makedirs(out_dir, exist_ok=True)

raw = np.linspace(-6, 6, 1000)
alpha_plot = 1.0  # show the curve for alpha=1; separately mark the alpha=0 output

defs = [
    {
        "name": "Gamma",
        "filename": "gamma.svg",
        "range_label": "[0, +inf)",
        "identity_label": "1.0",
        "formula_label": "alpha * max_gamma * sigmoid(raw)",
        "f": lambda r: alpha_plot * 3.0 / (1.0 + np.exp(-r)),
        "alpha0": 0.0,  # from the provided formula
        "bounds": [0.0],
        "notes": "Upper bound is unbounded by the stated range; plotted with max_gamma=3 for illustration.",
    },
    {
        "name": "Sharp",
        "filename": "sharp.svg",
        "range_label": "[0, +inf)",
        "identity_label": "1.0",
        "formula_label": "alpha * max_sharp * sigmoid(raw)",
        "f": lambda r: alpha_plot * 15.0 / (1.0 + np.exp(-r)),
        "alpha0": 0.0,  # from the provided formula
        "bounds": [0.0],
        "notes": "Upper bound is unbounded by the stated range; plotted with max_sharp=15 for illustration.",
    },
    {
        "name": "Wb",
        "filename": "wb.svg",
        "range_label": "(-inf, +inf)",
        "identity_label": "0.0",
        "formula_label": "sigmoid(alpha * raw)",
        "f": lambda r: 1.0 / (1.0 + np.exp(-(alpha_plot * r))),
        "alpha0": 0.5,
        "bounds": [],
        "notes": "Stated range is unbounded, but the provided smooth constraint maps into (0, 1).",
    },
    {
        "name": "Exposure",
        "filename": "exposure.svg",
        "range_label": "(-inf, +inf)",
        "identity_label": "0.0",
        "formula_label": "5 * tanh(alpha * raw)",
        "f": lambda r: 5.0 * np.tanh(alpha_plot * r),
        "alpha0": 0.0,
        "bounds": [5.0, -5.0],
        "notes": "Stated range is unbounded, but the provided smooth constraint maps into (-5, 5).",
    },
    {
        "name": "Bright",
        "filename": "bright.svg",
        "range_label": "[0, 1]",
        "identity_label": "0.0",
        "formula_label": "sigmoid(alpha * raw)",
        "f": lambda r: 1.0 / (1.0 + np.exp(-(alpha_plot * r))),
        "alpha0": 0.5,
        "bounds": [0.0, 1.0],
        "notes": "Provided smooth constraint puts alpha=0 at 0.5, not at the stated identity 0.0.",
    },
    {
        "name": "Contrast",
        "filename": "contrast.svg",
        "range_label": "[0, +inf)",
        "identity_label": "1.0",
        "formula_label": "exp(alpha * scale * tanh(raw))",
        "f": lambda r: np.exp(alpha_plot * 1.1 * np.tanh(r)),
        "alpha0": 1.0,
        "bounds": [0.0],
        "notes": "Plotted with scale=1.1 for illustration; lower bound is 0, upper bound unbounded.",
    },
    {
        "name": "Saturation",
        "filename": "saturation.svg",
        "range_label": "[0, +inf)",
        "identity_label": "1.0",
        "formula_label": "exp(alpha * scale * tanh(raw))",
        "f": lambda r: np.exp(alpha_plot * 2.3 * np.tanh(r)),
        "alpha0": 1.0,
        "bounds": [0.0],
        "notes": "Plotted with scale=2.3 for illustration; lower bound is 0, upper bound unbounded.",
    },
    {
        "name": "Bw",
        "filename": "bw.svg",
        "range_label": "(-inf, +inf)",
        "identity_label": "0.0",
        "formula_label": "sigmoid(alpha * raw)",
        "f": lambda r: 1.0 / (1.0 + np.exp(-(alpha_plot * r))),
        "alpha0": 0.5,
        "bounds": [],
        "notes": "Stated range is unbounded, but the provided smooth constraint maps into (0, 1).",
    },
    {
        "name": "Tone",
        "filename": "tone.svg",
        "range_label": "(-inf, +inf)",
        "identity_label": "1.0",
        "formula_label": "exp(alpha * scale * tanh(raw))",
        "f": lambda r: np.exp(alpha_plot * 1.1 * np.tanh(r)),
        "alpha0": 1.0,
        "bounds": [],
        "notes": "Stated range is unbounded, but the provided smooth constraint is strictly positive.",
    },
    {
        "name": "Hue",
        "filename": "hue.svg",
        "range_label": "[-pi, pi]",
        "identity_label": "0.0",
        "formula_label": "pi * tanh(alpha * raw)",
        "f": lambda r: math.pi * np.tanh(alpha_plot * r),
        "alpha0": 0.0,
        "bounds": [math.pi, -math.pi],
        "notes": "Finite bounds are exactly ±pi.",
    },
    {
        "name": "Color",
        "filename": "color.svg",
        "range_label": "(-inf, +inf)",
        "identity_label": "1.0",
        "formula_label": "exp(alpha * scale * tanh(raw))",
        "f": lambda r: np.exp(alpha_plot * 1.1 * np.tanh(r)),
        "alpha0": 1.0,
        "bounds": [],
        "notes": "Stated range is unbounded, but the provided smooth constraint is strictly positive.",
    },
    {
        "name": "Blur",
        "filename": "blur.svg",
        "range_label": "(0, +inf)",
        "identity_label": "Near 0",
        "formula_label": "alpha * max_blur * sigmoid(raw) + 1e-6",
        "f": lambda r: alpha_plot * 10.0 / (1.0 + np.exp(-r)) + 1e-6,
        "alpha0": 1e-6,
        "bounds": [1e-6],
        "notes": "Plotted with max_blur=10 for illustration; lower bound is 1e-6 and upper bound follows max_blur for alpha=1.",
    },
]

saved = []

for d in defs:
    y = d["f"](raw)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(raw, y, label=f"curve for alpha={alpha_plot}")
    ax.axhline(d["alpha0"], linestyle="--", label="value when alpha=0")

    finite_bounds = d["bounds"]
    for i, b in enumerate(finite_bounds):
        ax.axhline(b, linestyle=":", label="bound" if i == 0 else None)

    ax.set_title(
        f'{d["name"]}\n{d["formula_label"]}\nRange: {d["range_label"]} | Identity: {d["identity_label"]}'
    )
    ax.set_xlabel("raw")
    ax.set_ylabel("output")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    # Add a small explanatory footer inside the figure area
    fig.text(0.01, 0.01, d["notes"], fontsize=8)

    path = os.path.join(out_dir, d["filename"])
    fig.savefig(path, format="svg", bbox_inches="tight")
    plt.close(fig)
    saved.append(path)

# Also create a simple HTML index for convenience
html_path = os.path.join(out_dir, "index.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><body><h1>Smooth constraint SVG plots</h1>\n")
    f.write("<p>Each plot shows the curve for alpha=1, a dashed line for the output when alpha=0, and dotted finite bounds where applicable.</p>\n")
    for d in defs:
        f.write(f"<h2>{d['name']}</h2>\n")
        f.write(f"<p>{d['formula_label']} | Range: {d['range_label']} | Identity: {d['identity_label']}</p>\n")
        f.write(f'<img src="{d["filename"]}" style="max-width:100%; border:1px solid #ccc;" />\n')
    f.write("</body></html>")

print("Created files:")
for p in saved:
    print(p)
print(html_path)


OSError: [Errno 30] Read-only file system: '/mnt/data'